# Aula 5 — Reconhecer padrões sem se enganar

**Ideia central:** Um classificador é tão bom quanto os descritores que recebe e a validação que o avalia. Se a mesma folha está no treino e no teste, o modelo não aprendeu a doença, aprendeu a folha.

**Ementa:** Representação e descrição • Espaço de características • k-NN, mínima distância, Bayes • Matriz de confusão • Validação por grupo

---

In [ ]:
import numpy as np
import pandas as pd
from skimage import io, color
from skimage.feature import graycomatrix, graycoprops
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import StratifiedKFold, LeaveOneGroupOut, cross_val_predict
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt

## 1. Extração de descritores

Descritores de cor (HSV) e textura (GLCM).

In [ ]:
def descritores(caminho):
    """Extrai descritores de cor e textura de um recorte de folha."""
    img = io.imread(caminho)[..., :3]
    hsv = color.rgb2hsv(img)
    
    # Cinza quantizado para GLCM (16 níveis)
    cinza = (color.rgb2gray(img) * 255).astype(np.uint8) // 16
    glcm = graycomatrix(cinza, distances=[1], angles=[0], 
                        levels=16, symmetric=True, normed=True)
    
    # Máscara de pixels marrons (lesões)
    marrom = ((hsv[..., 0] > 8/360) & (hsv[..., 0] < 42/360) & 
              (hsv[..., 2] < 0.62) & (hsv[..., 1] > 0.3))
    
    return {
        "matiz": hsv[..., 0].mean(),
        "sat": hsv[..., 1].mean(),
        "brilho": hsv[..., 2].mean(),
        "marrom": marrom.mean(),
        "contraste": graycoprops(glcm, "contrast")[0, 0],
        "desvio": cinza.std()
    }

## 2. Carregar dataset

Estrutura esperada: CSV com colunas `arquivo, classe, fazenda, folha`

In [ ]:
# Exemplo com dados sintéticos (substitua pelo seu dataset)
# meta = pd.read_csv("metadados.csv")
# X = pd.DataFrame([descritores(a) for a in meta.arquivo])
# y = meta.classe

# Dados sintéticos para demonstração
np.random.seed(42)
n = 120

# Simula 3 classes: sadia, ferrugem, mancha
X = pd.DataFrame({
    "matiz": np.concatenate([np.random.normal(0.33, 0.02, 40),  # verde
                             np.random.normal(0.08, 0.03, 40),  # marrom
                             np.random.normal(0.15, 0.04, 40)]), # amarelado
    "sat": np.random.rand(n) * 0.5 + 0.3,
    "brilho": np.random.rand(n) * 0.4 + 0.4,
    "marrom": np.concatenate([np.random.rand(40) * 0.05,
                              np.random.rand(40) * 0.3 + 0.2,
                              np.random.rand(40) * 0.15 + 0.1]),
    "contraste": np.random.rand(n) * 10 + 2,
    "desvio": np.random.rand(n) * 20 + 10
})
y = np.array(["sadia"] * 40 + ["ferrugem"] * 40 + ["mancha"] * 40)
fazenda = np.array(["A"] * 30 + ["B"] * 30 + ["C"] * 30 + ["D"] * 30)

print(f"Amostras: {len(X)}")
print(f"Classes: {np.unique(y)}")
print(f"Fazendas: {np.unique(fazenda)}")

## 3. Visualizar espaço de características

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 5))

cores = {"sadia": "green", "ferrugem": "orange", "mancha": "gold"}
for classe in np.unique(y):
    mask = y == classe
    ax[0].scatter(X.loc[mask, "matiz"], X.loc[mask, "marrom"], 
                  c=cores[classe], label=classe, alpha=0.7)
ax[0].set_xlabel("Matiz média")
ax[0].set_ylabel("% pixels marrons")
ax[0].legend()
ax[0].set_title("Espaço de características")

# Por fazenda
marcadores = {"A": "o", "B": "s", "C": "^", "D": "D"}
for faz in np.unique(fazenda):
    mask = fazenda == faz
    ax[1].scatter(X.loc[mask, "matiz"], X.loc[mask, "marrom"],
                  marker=marcadores[faz], label=f"Fazenda {faz}", alpha=0.7)
ax[1].set_xlabel("Matiz média")
ax[1].set_ylabel("% pixels marrons")
ax[1].legend()
ax[1].set_title("Colorido por fazenda")

plt.tight_layout()
plt.show()

## 4. Comparar validações: aleatória vs por grupo

In [ ]:
print("Classificador      | CV Aleatória | Leave-One-Farm-Out")
print("-" * 55)

for nome, clf in [("k-NN (k=5)", KNeighborsClassifier(5)), 
                  ("Naive Bayes", GaussianNB())]:
    modelo = make_pipeline(StandardScaler(), clf)
    
    # Validação cruzada aleatória
    p_aleat = cross_val_predict(modelo, X, y, 
                                 cv=StratifiedKFold(4, shuffle=True, random_state=0))
    
    # Validação deixando uma fazenda de fora
    p_grupo = cross_val_predict(modelo, X, y, 
                                 cv=LeaveOneGroupOut(), groups=fazenda)
    
    acc_aleat = (p_aleat == y).mean()
    acc_grupo = (p_grupo == y).mean()
    
    print(f"{nome:<18} |    {acc_aleat:.3f}     |       {acc_grupo:.3f}")

print("\n" + "=" * 55)
print("Se a CV aleatória for muito maior que a por grupo,")
print("o modelo pode estar aprendendo características da fazenda,")
print("não da doença!")

## 5. Matriz de confusão e métricas

In [ ]:
# Usando validação por grupo (mais honesta)
modelo = make_pipeline(StandardScaler(), KNeighborsClassifier(5))
p_grupo = cross_val_predict(modelo, X, y, cv=LeaveOneGroupOut(), groups=fazenda)

print("Relatório de classificação (Leave-One-Farm-Out):")
print(classification_report(y, p_grupo))

print("\nMatriz de confusão:")
print(confusion_matrix(y, p_grupo))

---

## Desafios

### Aquecimento
Encontre o par de descritores que dá a maior acurácia com mínima distância. Explique, olhando a dispersão, por que funciona.

### Mão na massa
Use o código num conjunto real (PlantVillage ou fotos da turma). Compare StratifiedKFold com GroupKFold usando a folha como grupo.

### Investigação
Proponha uma normalização de cor que reduza o efeito fazenda. Mostre a acurácia leave-one-farm-out antes e depois.

In [ ]:
# Espaço para os desafios


---

## Leitura recomendada

- FACELI, K. et al. *Inteligência artificial: uma abordagem de aprendizagem de máquina*. LTC, 2011.
- GONZALEZ, R. C.; WOODS, R. E. Cap. 11 e 12.
- ROBERTS, D. R. et al. Cross-validation strategies for data with temporal, spatial, hierarchical structure. Ecography, 2017.